In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import equinox as eqx
import matplotlib.pyplot as plt
import optax
from typing import List
from jax.example_libraries import optimizers

In [ ]:
N_DOF_FD = 200
N_COLLOCATION_POINTS = 200
LEARNING_RATE = .5e-3
N_OPTIMIZATION_EPOCHS = 50_000
BC_LOSS_WEIGHT = 1.0

In [ ]:
key = jax.random.PRNGKey(42)


In [ ]:
# Our PINN is a coordinate network in the form of a MLP, mapping to scalar to scalar values
key, init_key = jr.split(key)
pinn = eqx.nn.MLP(
    in_size="scalar",
    out_size="scalar",
    width_size=30,
    depth=5,
    activation=jax.nn.sigmoid,
    key=init_key,
)

In [ ]:
pinn(.2)

In [ ]:
mesh_full = jnp.linspace(0.0, 10.0, N_DOF_FD + 2)

In [ ]:
mesh_interior = mesh_full[1:-1]
print(mesh_interior)

In [ ]:
rhs_function = lambda x: .1*( \
    -(4*jnp.pi**2*(x-2.5)**2-2*jnp.pi) / (jnp.exp(jnp.pi*(x-2.5)**2)) \
    -(8*jnp.pi**2*(x-7.5)**2-4*jnp.pi) / (jnp.exp(jnp.pi*(x-7.5)**2)) \
    )

In [ ]:
rhs_function(.5)

In [ ]:
rhs_evaluated = rhs_function(mesh_interior)
print(rhs_evaluated)

In [ ]:
dx = mesh_interior[1] - mesh_interior[0]
A = jnp.diag(jnp.ones(N_DOF_FD - 1), -1) + jnp.diag(jnp.ones(N_DOF_FD - 1), 1) - jnp.diag(2 * jnp.ones(N_DOF_FD), 0)
A /= dx**2

In [ ]:
finite_difference_solution = jnp.linalg.solve(A, -rhs_evaluated)/175

In [ ]:
wrap_bc = lambda u: jnp.pad(u, (1, 1), mode="constant")

In [ ]:
plt.plot(mesh_full, wrap_bc(rhs_evaluated), label="Forcing Function")
plt.plot(mesh_full, wrap_bc(finite_difference_solution), label="Finite Difference solution")
plt.plot(mesh_full, jax.vmap(pinn)(mesh_full), label="Initial PINN solution")
plt.legend()
plt.grid()

In [ ]:
def pde_residuum(network, x):
    return jax.grad(jax.grad(network))(x) + rhs_function(x)

In [ ]:
pde_residuum(pinn, 0.8)

In [ ]:
key, sampling_key = jr.split(key)
collocation_points = jr.uniform(sampling_key, (N_COLLOCATION_POINTS, ), minval=0.0 + 0.001, maxval=10.0 - 0.001)

def loss_fn(network):
    pde_residuum_at_collocation_points = jax.vmap(pde_residuum, in_axes=(None, 0))(network, collocation_points)
    pde_loss_contribution = 0.5 * jnp.mean(jnp.square(pde_residuum_at_collocation_points))

    left_bc_residuum = network(0.0) - 0.0
    right_bc_residuum = network(10.0) - 0.0
    bc_residuum_contribution = 0.5 * jnp.mean(jnp.square(left_bc_residuum)) + 0.5 * jnp.mean(jnp.square(right_bc_residuum))

    total_loss = pde_loss_contribution + BC_LOSS_WEIGHT * bc_residuum_contribution

    return total_loss

In [ ]:
# Training loop
lr = optimizers.exponential_decay(1e-3, decay_steps=2000, decay_rate=0.9)
optimizer = optax.adam(lr)
#optimizer = optax.adam(LEARNING_RATE)
opt_state = optimizer.init(eqx.filter(pinn, eqx.is_array))

@eqx.filter_jit
def make_step(network, state):
    loss, grad = eqx.filter_value_and_grad(loss_fn)(network)
    updates, new_state = optimizer.update(grad, state, network)
    new_network = eqx.apply_updates(network, updates)
    return new_network, new_state, loss

loss_history = []
for epoch in range(N_OPTIMIZATION_EPOCHS):
    pinn, opt_state, loss = make_step(pinn, opt_state)
    loss_history.append(loss)
    if epoch % 100 == 0:
        print(f"Epoch: {epoch}, loss: {loss}")

In [ ]:
plt.plot(loss_history)
plt.yscale("log")

In [ ]:
plt.plot(mesh_full, wrap_bc(finite_difference_solution)*175, label="Finite Difference solution")
plt.plot(mesh_full, jax.vmap(pinn)(mesh_full), label="Final PINN solution")
plt.legend()
plt.grid()